In [ ]:
import pandas as pd

# Load datasets
delhi_df = pd.read_csv("../data/Delhi.csv")
mumbai_df = pd.read_csv("../data/Mumbai.csv")

# Show first rows
display(delhi_df.head())
display(mumbai_df.head())

In [ ]:
print("Delhi Shape:", delhi_df.shape)
print("Mumbai Shape:", mumbai_df.shape)

In [ ]:
print(delhi_df.columns)
print(mumbai_df.columns)

In [ ]:
delhi_df.info()

In [ ]:
mumbai_df.info()

In [ ]:
delhi_df["City"] = "Delhi"
mumbai_df["City"] = "Mumbai"

In [ ]:
df = pd.concat([delhi_df, mumbai_df], ignore_index=True)

In [ ]:
print(df.shape)

df.head()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
data = df.copy()

In [ ]:
data.isnull().sum().sort_values(ascending=False)

In [ ]:
data.columns

In [ ]:
data.drop(columns=["JoggingTrack", "RainWaterHarvesting", "MultipurposeRoom","StaffQuarter","ShoppingMall","GolfCourse","Stadium","VaastuCompliant"], inplace=True)

In [ ]:
data.columns

In [ ]:
data.drop(columns=["Intercom","Hospital","PowerBackup"], inplace=True)

In [ ]:
data.columns

In [ ]:
data["Price"] = (
    data["Price"]
    .astype(str)
    .str.replace(",", "")
    .str.replace("₹", "")
)

data["Price"] = pd.to_numeric(data["Price"])

In [ ]:
data.info()

In [ ]:
data["Area"].head(20)

In [ ]:
def clean_area(x):
    x = str(x)

    # Handle ranges
    if "-" in x:
        nums = x.split("-")
        return (float(nums[0]) + float(nums[1])) / 2

    # Extract numeric part
    nums = "".join(ch for ch in x if ch.isdigit() or ch == ".")

    return float(nums) if nums else None

In [ ]:
data["Area"] = data["Area"].apply(clean_area)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.boxplot(data["Price"])
plt.title("Price Outliers")
plt.show()

In [ ]:
Q1 = data["Price"].quantile(0.25)
Q3 = data["Price"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

data = data[
    (data["Price"] >= lower) &
    (data["Price"] <= upper)
]

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(data["Price"])
plt.title("Price Outliers")
plt.show()

In [ ]:
data.loc[:, "Price_per_sqft"] = data["Price"] / data["Area"]

In [ ]:
def size_category(area):
    if area < 800:
        return "Small"
    elif area < 1500:
        return "Medium"
    elif area < 2500:
        return "Large"
    else:
        return "Luxury"

data.loc[:, "Size_category"] = data["Area"].apply(size_category)

In [ ]:
data.drop(columns=["ClubHouse"], inplace=True)

In [ ]:
data.columns

In [ ]:
data.select_dtypes(include="object").columns

In [ ]:
data["Luxury_Score"] = (
    data["Area"] * 0.35 +
    data["No. of Bedrooms"] * 0.25 +
    data["SwimmingPool"] * 0.10 +
    data["Gymnasium"] * 0.10 +
    data["LiftAvailable"] * 0.05 +
    data["AC"] * 0.05 +
    data["CarParking"] * 0.10
)

In [ ]:
def luxury_category(score):
    if score < 800:
        return "Basic"
    elif score < 1500:
        return "Premium"
    else:
        return "Luxury"

data["Luxury_Category"] = data["Luxury_Score"].apply(luxury_category)

In [ ]:
data.drop(columns=["WashingMachine","BED","TV","Sofa", "Wardrobe","DiningTable","Microwave","Gasconnection","ATM", "Cafeteria","IndoorGames"], inplace=True)

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
data.columns = (
    data.columns
    .str.replace(" ", "_")
    .str.replace(".", "")
    .str.replace("/", "_")
    .str.replace("'", "")
)

In [ ]:
data.columns

In [ ]:
data["Location"].value_counts()

In [ ]:
location_counts = data["Location"].value_counts()

rare_locations = location_counts[location_counts < 10].index

In [ ]:
data["Location"] = data["Location"].replace(
    rare_locations,
    "Other"
)

In [ ]:
data["Location"].nunique()

In [ ]:
data.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,5))

sns.histplot(data["Price"], bins=50)

plt.title("House Price Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="City",
    y="Price",
    data=data
)

plt.title("Delhi vs Mumbai Prices")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    x="Area",
    y="Price",
    hue="City",
    data=data
)

plt.title("Area vs Price")

plt.show()

In [ ]:
data.groupby("City")["Price_per_sqft"].mean()

In [ ]:
sns.countplot(
    x="Luxury_Category",
    hue="City",
    data=data
)

plt.title("Luxury Property Distribution")

plt.show()

In [ ]:
top_locations = (
    data.groupby("Location")["Price"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

top_locations.plot(
    kind="bar",
    figsize=(12,5)
)

plt.title("Top Expensive Locations")

plt.show()

In [ ]:
sns.boxplot(
    x="SwimmingPool",
    y="Price",
    data=data
)

plt.title("Swimming Pool vs Price")

plt.show()

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    data.select_dtypes(include="number").corr(),
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
columns_to_drop = [
    "MaintenanceStaff",
    "LandscapedGardens",
    "SportsFacility",
    "School",
    "Wifi",
    "Childrensplayarea",
    "Resale"
]

data.drop(columns=columns_to_drop, inplace=True)

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    data.select_dtypes(include="number").corr(),
    cmap="coolwarm"
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
data.select_dtypes(include="object").columns

In [ ]:
data_encoded = pd.get_dummies(
    data,
    drop_first=True
)

In [ ]:
data_encoded.shape

In [ ]:
y = data_encoded["Price"]

In [ ]:
X = data_encoded.drop("Price", axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr_model = LinearRegression()

In [ ]:
lr_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
y_pred = lr_model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

In [ ]:
mse = mean_squared_error(
    y_test,
    y_pred
)

In [ ]:
import numpy as np

rmse = np.sqrt(mse)

In [ ]:
r2 = r2_score(
    y_test,
    y_pred
)

In [ ]:
print("MAE :", mae)
print("MSE :", mse)
print("RMSE :", rmse)
print("R2 Score :", r2)

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_
})

coefficients.sort_values(
    by="Coefficient",
    ascending=False
).head(20)

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison.head(20)

In [ ]:
import joblib

joblib.dump(
    lr_model,
    "../models/linear_regression_model.pkl"
)

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

joblib.dump(
    X.columns.tolist(),
    "../models/model_columns.pkl"
)

In [ ]:
import joblib

city_locations = {
    "Delhi": sorted(
        data[data["City"] == "Delhi"]["Location"].unique()
    ),

    "Mumbai": sorted(
        data[data["City"] == "Mumbai"]["Location"].unique()
    )
}

joblib.dump(
    city_locations,
    "../models/city_locations.pkl"
)